In [13]:
import streamlit as st
import plotly.express as px
import pandas as pd
import geopandas as gpd
import folium
#from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from pathlib import Path
from streamlit_folium import st_folium

# ─── CHARGEMENT DONNÉES ───────────────────────────────────────────
df = pd.read_csv("../data/etablissements_occitanie.csv", sep=';',dtype={"departement": str, "code_insee": str})
df_communes = pd.read_csv("../data/communes-france-2025.csv", sep=",", encoding="utf-8", dtype=str)
df_communes_occitanie = df_communes[df_communes['reg_nom'] == 'Occitanie']
df_soin = pd.read_csv("../data/soins_limit.csv", sep=';', encoding="utf-8", dtype={"departement": str,"code_insee": str})
df_urgences = df_soin[df_soin['libactivite'].str.contains("urgence", case=False, na=False)]
df_distances = pd.read_csv("../data/distances_communes_urgence_occitanie.csv")
df_equipements = pd.read_csv("../data/equipements_occitanie.csv", sep=';', encoding="utf-8", dtype={"departement": str,"code_insee": str})

#jointure de soins et communes
df_soin_communes = pd.merge(df_soin, df_communes, on="code_insee", how="inner")

In [14]:
df_equipements.head()

,nofinesset,de_equipement,libde_equipement,libta_equipement,capinstot,nofinessej,rs,rslongue,commune,departement,libdepartement,categetab,libcategetab,coordyet,coordxet,longitude,latitude,groupe_equipement,groupe,code_insee
0,90003922,507,Hébergement médico soc personnes en difficulté...,Hébergement Complet Internat,14.0,310026133,UCRM - ACT - ANTENNE PAMIERS,NaN,225,09,ARIEGE,165,Appartement de Coordination Thérapeutique (A.C...,6225284.3,587186.9,1.614832,43.117920,Hébergement médico soc,Coordination / Administration,09225
1,110003068,507,Hébergement médico soc personnes en difficulté...,Hébergement Complet Internat,16.0,750015968,ACT GROUPE SOS SOLIDARITES,ACT SOS HABITAT ET SOINS CARCASSONNE,69,11,AUDE,165,Appartement de Coordination Thérapeutique (A.C...,6236151.6,648325.9,2.364397,43.222698,Hébergement médico soc,Coordination / Administration,11069
2,120007562,507,Hébergement médico soc personnes en difficulté...,Hébergement Complet Internat,6.0,120783931,ACT VILLEFRANCHE DE ROUERGUE,APPARTEMENT DE COORDINATION THERAPEUTIQUE,300,12,AVEYRON,165,Appartement de Coordination Thérapeutique (A.C...,6361484.2,623238.6,2.037044,44.348176,Hébergement médico soc,Coordination / Administration,12300
3,300003399,507,Hébergement médico soc personnes en difficulté...,Hébergement Complet Internat,18.0,750015968,ACT LOU CANTOU NIMES,APPARTEMENTS DE COORDINATION THERAPEUTIQUE LOU...,189,30,GARD,165,Appartement de Coordination Thérapeutique (A.C...,6305146.3,809570.4,4.362277,43.836821,Hébergement médico soc,Coordination / Administration,30189
4,300012259,507,Hébergement médico soc personnes en difficulté...,Hébergement de Nuit Eclaté,9.0,300786324,ACT AGFAS ALES,ACT AGFAS ALES,7,30,GARD,165,Appartement de Coordination Thérapeutique (A.C...,6336873.3,787150.3,4.089009,44.125456,Hébergement médico soc,Coordination / Administration,30007


In [8]:
df_soin_communes.head()

,numero_finess_etablissement,rsej,activite,libactivite,forme,libforme,numero finess etablissement juridique,raison_sociale,rslongue,code_commune,...,code_unite_urbaine,taille_unite_urbaine,population,superficie_hectare,superficie_km2,densite,latitude_centre,longitude_centre,grille_densite,grille_densite_texte
0,130044977,APHM DIRECTION GENERALE,14,Médecine d'urgence,14,Non saisonnier,130786049,APHM ANTENNE SMUR DE MARIGNANE,APHM ANTENNE SMUR DE MARIGNANE SITE SDIS,54,...,759,7.0,33003,2355,24,1401.0,43.417,5.212,2,Centres urbains intermédiaires
1,130045016,APHM DIRECTION GENERALE,14,Médecine d'urgence,14,Non saisonnier,130786049,APHM ANTENNE SMUR DE MARIGNANE,APHM ANTENNE SMUR MARIGNANE SITE HELICO SECURI...,54,...,759,7.0,33003,2355,24,1401.0,43.417,5.212,2,Centres urbains intermédiaires
2,300016680,CHU NIMES,4,Psychiatrie,3,Hospitalisation à temps partiel de jour,300780038,HJ PIJ CMPEA OUEST MONTAURY CHU NIMES,HJ PSY INFANTO JUVENILE ET CMPEA OUEST MONTAUR...,189,...,30601,6.0,148104,16118,161,919.0,43.845,4.348,1,Grands centres urbains
3,300016698,CHU NIMES,4,Psychiatrie,3,Hospitalisation à temps partiel de jour,300780038,HJ PIJ CMPEA VAUVERT CHU NIMES,HJ PSY INFANTO JUVENILE CMPEA VAUVERT,341,...,30302,3.0,11774,11052,111,107.0,43.626,4.306,2,Centres urbains intermédiaires
4,300017910,CHU NIMES,4,Psychiatrie,3,Hospitalisation à temps partiel de jour,300780038,HJ PSY GEN BOURDALOUE CHU NIMES,HJ PSY ADULTES BOURDALOUE CHU NIMES,189,...,30601,6.0,148104,16118,161,919.0,43.845,4.348,1,Grands centres urbains


In [12]:
df_soin.head()

,numero_finess_etablissement,rsej,activite,libactivite,forme,libforme,numero finess etablissement juridique,raison_sociale,rslongue,code_commune,departement,libelle_departement,categetab,categorie,coordyet,coordxet,longitude,latitude,groupe_activites,code_insee
0,130044977,APHM DIRECTION GENERALE,14,Médecine d'urgence,14,Non saisonnier,130786049,APHM ANTENNE SMUR DE MARIGNANE,APHM ANTENNE SMUR DE MARIGNANE SITE SDIS,54,13,BOUCHES DU RHONE,101,Centre Hospitalier Régional (C.H.R.),6260484.0,879240.3,5.212552,43.420867,Médecine d'urgence,13054
1,130044985,APHM DIRECTION GENERALE,14,Médecine d'urgence,14,Non saisonnier,130786049,APHM ANTENNE SMUR DE LOUVAIN,APHM ANTENNE SMUR DE LOUVAIN SITE BMPM,208,13,BOUCHES DU RHONE,101,Centre Hospitalier Régional (C.H.R.),6244839.0,894134.9,5.390459,43.276279,Médecine d'urgence,13208
2,130044993,APHM DIRECTION GENERALE,14,Médecine d'urgence,14,Non saisonnier,130786049,APHM ANTENNE SMUR DE PLOMBIERES,APHM ANTENNE SMUR DE PLOMBIERES SITE BMPM,203,13,BOUCHES DU RHONE,101,Centre Hospitalier Régional (C.H.R.),6249453.2,893151.8,5.380073,43.318035,Médecine d'urgence,13203
3,130045008,APHM DIRECTION GENERALE,14,Médecine d'urgence,14,Non saisonnier,130786049,APHM ANTENNE SMUR D'ENDOUME,APHM ANTENNE SMUR D'ENDOUME SITE BMPM,207,13,BOUCHES DU RHONE,101,Centre Hospitalier Régional (C.H.R.),6246007.2,891763.1,5.361701,43.287425,Médecine d'urgence,13207
4,130045016,APHM DIRECTION GENERALE,14,Médecine d'urgence,14,Non saisonnier,130786049,APHM ANTENNE SMUR DE MARIGNANE,APHM ANTENNE SMUR MARIGNANE SITE HELICO SECURI...,54,13,BOUCHES DU RHONE,101,Centre Hospitalier Régional (C.H.R.),6261318.0,878895.8,5.208589,43.428454,Médecine d'urgence,13054


In [10]:
departements_occitanie = ["09","11", "12", "30","31", "32", "34", "46", "48","65", "66", "81", "82",  ]
df_soin_occitanie = df_soin[df_soin["departement"].isin(departements_occitanie)]

In [11]:
df_soin_occitanie.head()

,numero_finess_etablissement,rsej,activite,libactivite,forme,libforme,numero finess etablissement juridique,raison_sociale,rslongue,code_commune,departement,libelle_departement,categetab,categorie,coordyet,coordxet,longitude,latitude,groupe_activites,code_insee
